# 04 · Cold start (Chapter 8)

Source: the local `ch8_rebuild_resumable.ipynb` (5090), including the retrieval-index fix (cells 22–23).

| Phase | Output |
|---|---|
| 0 drug-disjoint parent graph | Table 8.1 (bucket sizes) |
| 1 class filter (total ≥ 4 → 211 classes) and line graphs | Table 8.2 |
| 2 E1.1–E1.5 | Tables 8.3, 8.4, 8.7 |
| 3 warm-start control E2.1–E2.4 | Table 8.5 |
| 4 frozen ChemBERTa nearest prototype | Table 8.6 |
| fix | E1.4 / E2.4 with a union-graph retrieval index (the rows the thesis reports) |

Every phase skips work already on disk, so re-running after a disconnect is safe.

**Reproducing the thesis drug split:** a fresh Phase 0 gives a different split (node order depends on Python's string-hash seed). Point `THESIS_PARENT_GRAPH` at the cached `graphs_v4/parent_graph.bin` from the thesis run; it is copied into the new output folder and nothing in the thesis folder is modified.

In [ ]:
DATA_ROOT = "<DATA_ROOT: e.g. /home/you/ddi_data>"   # PLACEHOLDER: thesis inputs (read only)
WORK_ROOT = "<WORK_ROOT: e.g. /home/you/ddi_runs>"   # PLACEHOLDER: new outputs
TEST_MODE = True     # True: small test subgraph from notebook 00, few epochs
                     # False: full thesis graph (needs about 64 GB of RAM)
N_TEST_DRUGS = 800   # must match the test graph built in notebook 00

In [ ]:
import os, sys
# The package is installed in the venv by setup/setup_local.sh (editable install).
# Run this notebook with the "ddi (5090)" kernel.
try:
    import ddi
except ImportError:
    raise SystemExit('ddi is not importable: run "bash setup/setup_local.sh" and select '
                     'the "ddi (5090)" kernel.')
REPO_DIR = os.path.abspath(os.path.join(os.path.dirname(ddi.__file__), "..", ".."))
print("python:", sys.executable)
print("ddi from:", os.path.dirname(ddi.__file__))

In [ ]:
import json, gc
import numpy as np
import torch
import dgl
import warnings
warnings.filterwarnings("ignore", message="Dataloader CPU affinity opt is not enabled")

from ddi.paths import get_paths
paths = get_paths(DATA_ROOT, WORK_ROOT)
paths.make_output_dirs()
paths.show()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| TEST_MODE:", TEST_MODE)

In [ ]:
from ddi.experiments import coldstart as CS
import shutil

RUN_TAG = f"test{N_TEST_DRUGS}" if TEST_MODE else "v4"
d = CS.ColdStartDirs(base=paths.coldstart_dir, tag=RUN_TAG)
THESIS_PARENT_GRAPH = os.path.join(paths.thesis_ch8_dir, "graphs_v4", "parent_graph.bin")   # PLACEHOLDER if elsewhere

CS.banner("PHASE 0")
if os.path.exists(d.parent):
    print("  cached")
elif TEST_MODE:
    test_drug = dgl.load_graphs(os.path.join(paths.test_graph_dir, f"test_drug_graph_noneg_{N_TEST_DRUGS}.bin"))[0][0]
    CS.phase0_from_drug_graph(d, test_drug)
elif os.path.exists(THESIS_PARENT_GRAPH):
    shutil.copy(THESIS_PARENT_GRAPH, d.parent)
    print("  copied thesis parent graph (thesis drug split)")
else:
    from ddi.data import download_raw_data
    from ddi.paths import RAW_DATA_GDRIVE_ID
    drug_db, event_db = download_raw_data(paths.drug_db, paths.event_db, RAW_DATA_GDRIVE_ID)
    print("  WARNING: fresh rebuild; the drug split will NOT match the thesis")
    CS.phase0_build_parent(d, drug_db, event_db)
CS.mark(d, "phase0")
CS.phase0_report(d)

In [ ]:
CS.banner("PHASE 1")
meta = CS.phase1(d)
CS.mark(d, "phase1")

In [ ]:
EPOCHS = dict(max_epochs=2, min_epochs=1) if TEST_MODE else {}
CS.banner("PHASE 2 - cold-start runs E1.1-E1.5")
r2 = CS.phase2(d, device, **EPOCHS)

In [ ]:
CS.banner("PHASE 3 - warm-start control E2.1-E2.4")
r3 = CS.phase3(d, device, **EPOCHS)

In [ ]:
CS.banner("PHASE 4 - ChemBERTa floor")
r4 = CS.phase4(d, device)

## Retrieval-index fix (E1.4, E2.4)
Re-evaluates the saved E1.4 / E2.4 checkpoints with an index covering every node of the evaluation graph. `oob` must be 0.

In [ ]:
CS.banner("FIX E1.4"); f14 = CS.fix_e1_4(d, device)
CS.banner("FIX E2.4"); f24 = CS.fix_e2_4(d, device)

In [ ]:
CS.banner("TABLES")
_ = CS.phase5_tables(d)
CS.mark(d, "phase5")
print("\nThesis Table 8.3, E1.3 (Setup A/B, CS1 then CS2): 0.251 0.334 0.055 0.079")